In [1]:
import pandas as pd, numpy as np, time, joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

# 1) Load and clean one file
def load(path, day):
    d = pd.read_csv(path, encoding='latin1', low_memory=False)
    d.columns = d.columns.str.strip()
    d = d.replace([np.inf, -np.inf], np.nan).dropna().drop_duplicates()
    d['Timestamp'] = pd.to_datetime(d['Timestamp'], errors='coerce')
    d = d.dropna(subset=['Timestamp'])
    d['day'] = day
    return d

wed = load('data_cic/Wednesday.csv', 'Wed')
wed = wed[wed['Label'] != 'Heartbleed']
fri = load('data_cic/FridayDDoS.csv', 'Fri')
data = pd.concat([wed, fri], ignore_index=True)
print("Combined:", data.shape)

# 2) Features (IDs and label are kept aside, not used for training)
id_cols = ['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Timestamp', 'Label', 'day']
feature_cols = [c for c in data.columns if c not in id_cols]
print("Features:", len(feature_cols))

# 3) Temporal split per class: first 70% of time -> train, last 30% -> test
tr_parts, te_parts = [], []
for lbl, g in data.groupby('Label'):
    g = g.sort_values('Timestamp')
    cut = int(len(g) * 0.7)
    tr_parts.append(g.iloc[:cut])
    te_parts.append(g.iloc[cut:])
tr = pd.concat(tr_parts)
te = pd.concat(te_parts)

le2 = LabelEncoder()
le2.fit(data['Label'])
Xtr, ytr = tr[feature_cols].astype('float32'), le2.transform(tr['Label'])
Xte, yte = te[feature_cols].astype('float32'), le2.transform(te['Label'])
print("Train:", Xtr.shape, "| Test:", Xte.shape)

# 4) Train XGBoost
start = time.time()
xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                    tree_method='hist', n_jobs=-1, random_state=42, eval_metric='mlogloss')
xgb.fit(Xtr, ytr)
print("Training time: %.1f s" % (time.time() - start))

# 5) Evaluate
px = xgb.predict(Xte)
print("\nAccuracy: %.4f | Macro F1: %.4f" % (accuracy_score(yte, px), f1_score(yte, px, average='macro')))
print(classification_report(yte, px, target_names=le2.classes_, zero_division=0))
print(confusion_matrix(yte, px))

# 6) Save model + a 20k-flow sample (with IPs) for the Streamlit demo
joblib.dump(xgb, 'xgb_model.joblib')
joblib.dump(le2, 'label_encoder.joblib')
joblib.dump(feature_cols, 'feature_cols.joblib')
demo = te.sample(20000, random_state=42).sort_values('Timestamp')
demo[['Source IP', 'Destination IP', 'Timestamp', 'Label'] + feature_cols].to_csv('demo_flows.csv', index=False)
print("\nSaved: xgb_model.joblib, label_encoder.joblib, feature_cols.joblib, demo_flows.csv")

Combined: (917088, 86)
Features: 79
Train: (641960, 79) | Test: (275128, 79)
Training time: 31.3 s

Accuracy: 0.9520 | Macro F1: 0.9436
                  precision    recall  f1-score   support

          BENIGN       1.00      0.92      0.96    161206
            DDoS       0.76      1.00      0.86     38408
   DoS GoldenEye       0.95      1.00      0.97      3088
        DoS Hulk       1.00      0.99      0.99     69037
DoS Slowhttptest       0.89      0.99      0.94      1650
   DoS slowloris       0.98      0.89      0.93      1739

        accuracy                           0.95    275128
       macro avg       0.93      0.96      0.94    275128
    weighted avg       0.96      0.95      0.95    275128

[[148903  12251      0     45      7      0]
 [    28  38380      0      0      0      0]
 [    10      0   3073      0      5      0]
 [   488     13    145  68374      0     17]
 [     6      0      0      0   1633     11]
 [     4      0      1      1    186   1547]]

Saved: xg